# Exploring Classification Performance

Let's load in any libraries we will use in this notebook. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

#import torch which has many of the functions to build deep learning models and to train them
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

#import torchvision, which was lots of functions for loading and working with image data
import torchvision
import torchvision.transforms as transforms

#this is a nice progress bar representation that will be good to measure progress during training
import tqdm

#for creating confusion matrices from predictions
from sklearn.metrics import ConfusionMatrixDisplay

# 1) Splitting into a training and validation dataset

As we did in Week 4 and 5, we're going to load the dataset with our 20 dog species. 
I'm loading the 'train' subset, but acting as if it is a 'trainval' subset that I need to split into 'train' and 'val'.

In [ ]:
imagenet_means = (0.485, 0.456, 0.406)
imagenet_stds = (0.229, 0.224, 0.225)

transform = transforms.Compose(
    [transforms.ToTensor(),
    transforms.Resize((224, 224)), 
     transforms.Normalize(imagenet_means, imagenet_stds)])

trainval_dataset = torchvision.datasets.ImageFolder('../Week_4/stanford_dogs_subset/train', transform = transform)

## 1a) Using stratify to split into train and val

In the week 2 and week 3 tutorial, we explored how to use [sklearn.train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) to split a dataset.

Complete the code below to create a train and val dataset, with a 70/30% split of the trainval_dataset.

In [ ]:
train_portion = 0.7
val_portion = 0.3





train_dataset = ....
val_dataset = ....

In [ ]:
num_classes = len(trainval_dataset.classes)
class_labels = trainval_dataset.classes

train_labels = [data[1] for data in train_dataset]
val_labels = [data[1] for data in val_dataset]

plt.hist([train_labels, val_labels], bins = 20, label = ['Train', 'Val']) 
plt.xlabel('Class Label')
plt.legend()
plt.xticks([i for i in range(num_classes)], class_labels, rotation=90)
plt.ylabel('Count')
plt.show()

## Consider: What could be the downside of measuring performance with this validation dataset?

## 1b) Validation set with fixed samples per class

An alternative approach might be to create the validation set with a fixed number of samples, leaving all remaining data for the training set.

Below, we're going to split the dataset into train/val, always having 20 samples in the validation set.

In [ ]:
val_samples = 20


train_dataset = ...
val_dataset = ...

In [ ]:
num_classes = len(trainval_dataset.classes)
class_labels = trainval_dataset.classes

train_labels = [data[1] for data in train_dataset]
val_labels = [data[1] for data in val_dataset]

plt.hist([train_labels, val_labels], bins = 20, label = ['Train', 'Val']) 
plt.xlabel('Class Label')
plt.legend()
plt.xticks([i for i in range(num_classes)], class_labels, rotation=90)
plt.ylabel('Count')
plt.show()

## Consider: How many samples should we have in the validation dataset?

# 2) Finding performance on the val dataset

We're going to explore how to find:
1. Confusion matrix
    1. Total
    2. Grouped classes
2. Identifying precision and recall of grouped classes
3. The confidence calibration curve
   
## Create our model

We're going to use one of the ResNet18 models that I trained during the Week 5 tutorial.

To create the model for testing, we should:
1. Create the model
2. Adapt the architecture if necessary
3. Load in the saved weights
4. Put the model in eval mode
5. (Optional) move the model to the GPU

In [ ]:
model = torchvision.models.resnet18()
model.fc = nn.Linear(model.fc.in_features, 20)
model.load_state_dict(torch.load('ResNet18_Frozen_LR0.001.pth', map_location = 'cpu'))
model.eval()


## Initialise the val dataloader

We're going to create with a batch size of 1, as this will make some of the precision/recall calculations easier later on.

In [ ]:
valloader = torch.utils.data.DataLoader(val_dataset, batch_size=1,
                                          shuffle=False, num_workers = 2)

## Creating a confusion matrix

As we saw in Week 2 and Week 3, we can use [ConfusionMatrixDisplay](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html) from sklearn.metrics to create a confusion matrix.

To use the function, we need to pass in: 
- the GT label for each sample
- the predicted label for each sample
- (optional) display labels for each label (i.e. a list of class strings)
- (optional) normalize over 'true' or 'pred' labels to account for class imbalance

Below, let's first test our model over the val dataset and collect the GT label and predicted label for each sample.

Now, we can use the [ConfusionMatrixDisplay.from_predictions()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions) function to create the confusion matrix.

## Confusion Matrix with Grouped Classes

Sometimes, it's important to check the performance of a model with grouped classes. This can be really useful when not all types of mistakes are equal (e.g. Maybe it's better to confuse an Afghan Hound for a Bloodhound, than to confuse it for a Chihuahua).

**Let's check: How well does my model perform at distinguishing between 'hound' dogs and all other species of dog?**

Below, we're going to group performance into classifying hound species versus other species. A hound species is any dog species with the word 'hound' in its name.

## Identifying Precision and Recall for Grouped Classes

Precision and recall allow us to nicely explain certain performance characteristics of a classifier for certain classes or groups of classes. To calculate, for a chosen 'positive' class, we need to find the number of:
- TP = When a positive class sample is correctly classified
- FP = When a different class sample is incorrectly classified as the 'positive' class
- FN = When a positive class sample is incorrectly classified as a different class

We can then calculate precision and recall with:
- $Precision = \frac{TP}{TP+FP}$
- $Recall = \frac{TP}{TP+FN}$

Let's use the values we've already collected for our binary confusion matrix to identify the precision and recall for the 'Hound' group of classes.

Below, calculate:
- What's the precision for positive class Hound?
- What's the recall for positive class Hound?
  

In [ ]:
class_names = ['Hound', 'Other']
pos_class = 0

tp = 0
fp = 0
fn = 0
#### Insert your code here that finds the TP, FP and FN counts for the 'Hound' positive class in the dataset

############################################################################################################



prec = tp/(tp+fp)
rec = tp/(tp+fn)

print('For the Hound Class')
print(f'Precision: {100.*prec :.1f}%, Recall: {100.*rec :.1f}%')

## Confidence Calibration Curves

As discussed in the Week 6 lecture, calibration curves are useful for understanding how well the confidence scores predicted by a classification model align with the actual accuracy of the model, which is critical in some applications where not just the label but also the uncertainty of the prediction is important. A well-calibrated model should have its predicted confidence probabilities close to the true probability that the prediction is correct.

To do this, we need to collect all our GT labels, predictions, and the confidence associated with each prediction.

**Note: This relies on class scores being converted to pseudo-probablities using the [torch.nn.function.softmax() function](https://pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html).**


Once completing the above code, you can run the next cell to see the confidence calibration curves.

## Consider: Is the model: well-calibrated, over-confident, or under-confident? Is it prone to giving a certain confidence more than others?

How would this inform the advice you give someone who wants to use the model?

In [ ]:
#create a variable that holds the confidence intervals we will check on a confidence calibration curve
conf_ranges = [[0, 10], [10, 20], [20, 30], [30, 40], [40, 50], [50, 60], [60, 70], [70, 80], [80, 90], [90, 100]] 

#convert our previously collected lists into numpy arrays so that we can easily manipulate them
all_pred_conf = np.array(all_confidences)
all_pred_class = np.array(all_pred)
all_gt_class = np.array(all_gt)

actual_accuracy = []
conf_level = []
conf_counts = []
for conf_int in conf_ranges:
    lower = conf_int[0]/100 #convert between 0-1
    upper = conf_int[1]/100 #convert between 0-1

    #create a mask that will collect predictions in the confidence interval -- it must be above the lower thresh AND below the upper thresh
    mask = (all_pred_conf >= lower) & (all_pred_conf < upper)
    
    #collect all predictions and GT data within the range using the mask
    preds = all_pred_class[mask]
    gt = all_gt_class[mask]
    
    #find the accuracy of this bin by checking how many correct/total
    correct = np.sum(preds == gt)
    total = len(preds)
    accuracy = correct/total
    actual_accuracy += [accuracy] #save the accuracy for this bin to plot later
    conf_level += [(upper + lower)/2] #this is the average confidence level for this confidence interval (not necessarily for the predictions in the bin though), we will use this for plotting later

    #how many samples in this bin?
    conf_counts += [len(preds)]


#Create a figure 
fig, ax = plt.subplots(2, 1, figsize = (5, 7))
ax[0].bar(conf_level, actual_accuracy, width = 0.09)
ax[0].plot([0, 1], [0, 1], 'r--') #our well-calibrated line
ax[0].set_xlabel('Confidence')
ax[0].set_ylabel('Accuracy')
ax[0].set_title('Confidence Calibration Curve')

ax[1].bar(conf_level, conf_counts, width = 0.09)
ax[1].set_xlabel('Confidence')
ax[1].set_ylabel('Count')

plt.savefig('Confidence_curve.png')
plt.show()